In [4]:
# importar funciones correspondientes
%run "4.1_frameworkAgent.ipynb"

In [ ]:
# CURSO #3
# se necesitan de clases y funciones base del framework (`ActionContext`, `Prompt`, `register_tool`, `Agent`, `Goal`, `PythonActionRegistry`, `PythonEnvironment`, `AgentFunctionCallingActionLanguage`, `generate_response`.)
# import json

In [5]:
# pide al LLM que responda en JSON validado
@register_tool()
def prompt_llm_for_json(action_context: ActionContext, schema: dict, prompt: str):
    """
    Hace que el LLM genere JSON en respuesta a un prompt. Usa siempre esta
    herramienta cuando necesites datos estructurados del LLM.

    Args:
        schema: JSON Schema que define la estructura esperada
        prompt: El prompt que se le envía al LLM

    Returns:
        Un diccionario que cumple con el schema, con la información extraída
    """
    generate_response = action_context.get("llm")

    # Intenta hasta 3 veces obtener un JSON válido
    for i in range(3):
        try:
            response = generate_response(Prompt(messages=[
                {"role": "system",
                 "content": f"Debes producir una salida que cumpla con el siguiente "
                            f"JSON schema:\n\n{json.dumps(schema, indent=4)}. "
                            f"Escribe tu JSON dentro de un bloque markdown ```json."},
                {"role": "user", "content": prompt}
            ]))

            # Si la respuesta viene dentro de un bloque ```json, lo extraemos
            if "```json" in response:
                start = response.find("```json")
                end = response.rfind("```")
                response = response[start + 7:end].strip()

            return json.loads(response)

        except Exception as e:
            if i == 2:  # en el último intento, propaga el error
                raise e
            print(f"Error generando la respuesta: {e}")
            print("Reintentando...")

In [6]:
# herramienta para extraer datos de un invoice
@register_tool(tags=["document_processing", "invoices"])
def extract_invoice_data(action_context: ActionContext, document_text: str) -> dict:
    """
    Extrae datos estandarizados de una factura a partir de texto plano.

    Usa un schema fijo y un prompt especializado para garantizar que la
    extracción sea consistente sin importar el formato original de la factura.

    Args:
        document_text: El contenido de texto de la factura a procesar

    Returns:
        Un diccionario con los datos extraídos en formato estandarizado
    """
    invoice_schema = {
        "type": "object",
        "required": ["invoice_number", "date", "total_amount"],
        "properties": {
            "invoice_number": {"type": "string"},
            "date": {"type": "string"},
            "total_amount": {"type": "number"},
            "vendor": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "address": {"type": "string"}
                }
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "number"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"}
                    }
                }
            }
        }
    }

    extraction_prompt = f"""
            Eres un experto analista de facturas. Extrae la información de la
            factura con precisión y minuciosidad. Presta especial atención a:
            - Número de factura (busca 'Invoice #', 'No.', 'Referencia', etc.)
            - Fechas (enfócate en la fecha de emisión o de factura)
            - Montos (asegúrate de capturar correctamente el monto total)
            - Line items (captura todos los cargos individuales)

            Detente y piensa paso a paso. Luego, extrae los datos de la
            siguiente factura:

            <invoice>
            {document_text}
            </invoice>
    """

    return prompt_llm_for_json(
        action_context=action_context,
        schema=invoice_schema,
        prompt=extraction_prompt
    )

In [7]:
# herrmaienta de almacenamiento
@register_tool(tags=["storage", "invoices"])
def store_invoice(action_context: ActionContext, invoice_data: dict) -> dict:
    """
    Guarda una factura en la "base de datos" de facturas. Si ya existe una
    factura con el mismo número, se actualiza.

    Args:
        invoice_data: Los datos de la factura ya procesados

    Returns:
        Un diccionario con el resultado del guardado y el número de factura
    """
    storage = action_context.get("invoice_storage", {})

    invoice_number = invoice_data.get("invoice_number")
    if not invoice_number:
        raise ValueError("Los datos de la factura deben incluir un número de factura")

    storage[invoice_number] = invoice_data

    return {
        "status": "success",
        "message": f"Factura {invoice_number} guardada",
        "invoice_number": invoice_number
    }

In [8]:
# ensamblar el agente
def create_invoice_agent():
    action_registry = PythonActionRegistry()
    environment = PythonEnvironment()

    goals = [
        Goal(
            name="Persona",
            description="Eres un Agente de Procesamiento de Facturas, especializado "
                        "en manejar y almacenar datos de facturas."
        ),
        Goal(
            name="Procesar Facturas",
            description="""
            Tu objetivo es procesar facturas extrayendo sus datos y
            almacenándolos correctamente. Para cada factura:
            1. Extrae toda la información importante: números, fechas, montos y line items
            2. Guarda los datos extraídos indexados por número de factura
            3. Confirma que el procesamiento fue exitoso
            4. Maneja cualquier error de forma adecuada
            """
        )
    ]

    return Agent(
        goals=goals,
        agent_language=AgentFunctionCallingActionLanguage(),
        action_registry=action_registry,
        generate_response=generate_response,
        environment=environment
    )

In [9]:
# test
invoice_text = """
    Invoice #4567
    Date: 2025-02-01
    Vendor: Tech Solutions Inc.
    Items:
      - Laptop - $1,200
      - External Monitor - $300
    Total: $1,500
"""

agent = create_invoice_agent()
response = agent.run(f"Procesa esta factura:\n\n{invoice_text}")
print(response)

Agent thinking...
Agent Decision: {"tool": "extract_invoice_data", "args": {"document_text": "\n    Invoice #4567\n    Date: 2025-02-01\n    Vendor: Tech Solutions Inc.\n    Items:\n      - Laptop - $1,200\n      - External Monitor - $300\n    Total: $1,500\n"}}
Action Result: {'tool_executed': True, 'result': {'invoice_number': '4567', 'date': '2025-02-01', 'total_amount': 1500, 'vendor': {'name': 'Tech Solutions Inc.'}, 'line_items': [{'description': 'Laptop', 'quantity': 1, 'unit_price': 1200, 'total': 1200}, {'description': 'External Monitor', 'quantity': 1, 'unit_price': 300, 'total': 300}]}, 'timestamp': '2026-08-29T22:07:31-0600'}
Agent thinking...
Agent Decision: {"tool": "store_invoice", "args": {"invoice_data": {"date": "2025-02-01", "invoice_number": "4567", "line_items": [{"description": "Laptop", "quantity": 1, "total": 1200, "unit_price": 1200}, {"description": "External Monitor", "quantity": 1, "total": 300, "unit_price": 300}], "total_amount": 1500, "vendor": {"name": "

In [ ]:
# This is the worflow
# piensa → elige herramienta → ejecuta → actualiza memoria → repite hasta terminar...